In [ ]:
# Cell 1: Install dependencies
!pip install pandas numpy scikit-learn matplotlib seaborn joblib

In [ ]:
# Cell 2: Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import joblib
import time

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay,
                             f1_score, accuracy_score, classification_report)
from sklearn.pipeline import make_pipeline

In [ ]:
# Cell 3: Load Data
# Upload your CSV files via the Colab file browser before running this cell

air_df     = pd.read_csv("air_data.csv")
alcohol_df = pd.read_csv("alcohol_data.csv")
coffee_df  = pd.read_csv("coffee_data.csv")
vinegar_df = pd.read_csv("vinegar_data.csv")
wine_df    = pd.read_csv("wine_data.csv")

df = pd.concat([air_df, alcohol_df, coffee_df, vinegar_df, wine_df], ignore_index=True)
print(f"Total samples: {len(df)}")
print(f"Class distribution:")
print(df["label"].value_counts())
print(f"Columns: {list(df.columns)}")

In [ ]:
# Cell 4: Feature Engineering
# Per-sensor gas resistance (8 features) + log-transformed versions (8 features) = 16 total
# Log transform compresses the large range of gas resistance values (1k - 1M ohms)

base_cols = [col for col in df.columns if "gas" in col and "index" not in col]

X = df[base_cols].ffill().bfill()
X = X.dropna()

# Add log-transformed features
for col in base_cols:
    X[f"{col}_log"] = np.log1p(X[col])

feature_cols = list(X.columns)
y = df["label"][X.index]

print(f"Features ({len(feature_cols)}): {feature_cols}")
print(f"Feature matrix shape: {X.shape}")
print(f"Class distribution:")
print(y.value_counts())

# Plot raw vs log distributions for sensor 0 as example
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for label in y.unique():
    axes[0].hist(X["sensor_0_gas"][y == label], bins=30, alpha=0.6, label=label)
    axes[1].hist(X["sensor_0_gas_log"][y == label], bins=30, alpha=0.6, label=label)
axes[0].set_title("sensor_0_gas (raw)")
axes[1].set_title("sensor_0_gas (log)")
for ax in axes:
    ax.legend(fontsize=8)
plt.suptitle("Effect of Log Transform on Sensor 0")
plt.tight_layout()
plt.savefig("log_transform_effect.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Cell 5: Split and Scale
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = MinMaxScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

labels = sorted(y.unique())
print(f"Train: {X_train_sc.shape}  Test: {X_test_sc.shape}")
print(f"Classes: {labels}")

In [ ]:
# Cell 6: KNN Classifier
knn = KNeighborsClassifier(n_neighbors=5, weights="distance", metric="euclidean")
knn.fit(X_train_sc, y_train)

knn_pred = knn.predict(X_test_sc)
knn_acc  = accuracy_score(y_test, knn_pred)

# Latency benchmark
start = time.perf_counter()
for _ in range(1000):
    knn.predict(X_test_sc[:1])
knn_latency_ms = (time.perf_counter() - start)  # total seconds for 1000 runs
knn_latency_ms = knn_latency_ms * 1000 / 1000   # ms per prediction

print(f"KNN Accuracy:       {knn_acc:.3f}")
print(f"KNN Latency:        {knn_latency_ms:.3f} ms per prediction")
print(classification_report(y_test, knn_pred, target_names=labels))

knn_pipeline = make_pipeline(MinMaxScaler(), KNeighborsClassifier(n_neighbors=5, weights="distance", metric="euclidean"))
knn_cv = cross_val_score(knn_pipeline, X, y, cv=5, scoring="accuracy")
print(f"KNN CV accuracy:    {knn_cv.mean():.3f} +/- {knn_cv.std():.3f}")
print(f"Per-fold:           {knn_cv.round(3)}")

cm = confusion_matrix(y_test, knn_pred, labels=labels)
ConfusionMatrixDisplay(cm, display_labels=labels).plot(cmap="Blues")
plt.title("KNN Confusion Matrix")
plt.savefig("confusion_matrix_knn.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Cell 7: Random Forest Classifier
rf = RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_leaf=2, random_state=42)
rf.fit(X_train_sc, y_train)

rf_pred = rf.predict(X_test_sc)
rf_acc  = accuracy_score(y_test, rf_pred)

# Latency benchmark
start = time.perf_counter()
for _ in range(1000):
    rf.predict(X_test_sc[:1])
rf_latency_ms = (time.perf_counter() - start) * 1000 / 1000

print(f"RF Accuracy:        {rf_acc:.3f}")
print(f"RF Latency:         {rf_latency_ms:.3f} ms per prediction")
print(classification_report(y_test, rf_pred, target_names=labels))

rf_pipeline = make_pipeline(MinMaxScaler(), RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_leaf=2, random_state=42))
rf_cv = cross_val_score(rf_pipeline, X, y, cv=5, scoring="accuracy")
print(f"RF CV accuracy:     {rf_cv.mean():.3f} +/- {rf_cv.std():.3f}")
print(f"Per-fold:           {rf_cv.round(3)}")

cm = confusion_matrix(y_test, rf_pred, labels=labels)
ConfusionMatrixDisplay(cm, display_labels=labels).plot(cmap="Greens")
plt.title("Random Forest Confusion Matrix")
plt.savefig("confusion_matrix_rf.png", dpi=150, bbox_inches="tight")
plt.show()

# Feature importance
importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
plt.figure(figsize=(12, 4))
importances.plot(kind="bar")
plt.title("Random Forest - Feature Importance")
plt.ylabel("Importance")
plt.tight_layout()
plt.savefig("feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()
print(importances.round(4))

In [ ]:
# Cell 8: SVM Classifier
svm = SVC(kernel="rbf", C=10, gamma="scale", probability=True, random_state=42)
svm.fit(X_train_sc, y_train)

svm_pred = svm.predict(X_test_sc)
svm_acc  = accuracy_score(y_test, svm_pred)

# Latency benchmark
start = time.perf_counter()
for _ in range(1000):
    svm.predict(X_test_sc[:1])
svm_latency_ms = (time.perf_counter() - start) * 1000 / 1000

print(f"SVM Accuracy:       {svm_acc:.3f}")
print(f"SVM Latency:        {svm_latency_ms:.3f} ms per prediction")
print(classification_report(y_test, svm_pred, target_names=labels))

svm_pipeline = make_pipeline(MinMaxScaler(), SVC(kernel="rbf", C=10, gamma="scale", probability=True, random_state=42))
svm_cv = cross_val_score(svm_pipeline, X, y, cv=5, scoring="accuracy")
print(f"SVM CV accuracy:    {svm_cv.mean():.3f} +/- {svm_cv.std():.3f}")
print(f"Per-fold:           {svm_cv.round(3)}")

cm = confusion_matrix(y_test, svm_pred, labels=labels)
ConfusionMatrixDisplay(cm, display_labels=labels).plot(cmap="Oranges")
plt.title("SVM Confusion Matrix")
plt.savefig("confusion_matrix_svm.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Cell 9: MLP Neural Network
# Architecture: 16 inputs -> 128 -> 64 -> 32 -> 5 outputs
# ReLU activation with early stopping to prevent overfitting
mlp = MLPClassifier(
    hidden_layer_sizes=(128, 64, 32),
    activation="relu",
    solver="adam",
    max_iter=500,
    early_stopping=True,
    validation_fraction=0.1,
    random_state=42
)
mlp.fit(X_train_sc, y_train)

mlp_pred = mlp.predict(X_test_sc)
mlp_acc  = accuracy_score(y_test, mlp_pred)

# Latency benchmark
start = time.perf_counter()
for _ in range(1000):
    mlp.predict(X_test_sc[:1])
mlp_latency_ms = (time.perf_counter() - start) * 1000 / 1000

print(f"MLP Accuracy:       {mlp_acc:.3f}")
print(f"MLP Latency:        {mlp_latency_ms:.3f} ms per prediction")
print(classification_report(y_test, mlp_pred, target_names=labels))

# Cross-validation
mlp_pipeline = make_pipeline(
    MinMaxScaler(),
    MLPClassifier(hidden_layer_sizes=(128, 64, 32), activation="relu",
                  solver="adam", max_iter=500, early_stopping=True,
                  validation_fraction=0.1, random_state=42)
)
mlp_cv = cross_val_score(mlp_pipeline, X, y, cv=5, scoring="accuracy")
print(f"MLP CV accuracy:    {mlp_cv.mean():.3f} +/- {mlp_cv.std():.3f}")
print(f"Per-fold:           {mlp_cv.round(3)}")

# Confusion matrix
cm = confusion_matrix(y_test, mlp_pred, labels=labels)
ConfusionMatrixDisplay(cm, display_labels=labels).plot(cmap="Reds")
plt.title("MLP Confusion Matrix")
plt.savefig("confusion_matrix_mlp.png", dpi=150, bbox_inches="tight")
plt.show()

# Loss curve
plt.figure(figsize=(8, 4))
plt.plot(mlp.loss_curve_, label="Training loss")
if mlp.best_loss_ is not None:
    plt.axhline(y=mlp.best_loss_, color="r", linestyle="--", label="Best loss")
plt.title("MLP Training Loss Curve")
plt.xlabel("Iteration")
plt.ylabel("Loss")
plt.legend()
plt.tight_layout()
plt.savefig("mlp_loss_curve.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Cell 10: Voting Ensemble (KNN + RF + SVM + MLP)
# Soft voting averages the predicted probabilities from all three models
ensemble = VotingClassifier(
    estimators=[("knn", knn), ("rf", rf), ("svm", svm), ("mlp", mlp)],
    voting="soft"
)
ensemble.fit(X_train_sc, y_train)

ens_pred = ensemble.predict(X_test_sc)
ens_acc  = accuracy_score(y_test, ens_pred)

# Latency benchmark
start = time.perf_counter()
for _ in range(1000):
    ensemble.predict(X_test_sc[:1])
ens_latency_ms = (time.perf_counter() - start) * 1000 / 1000

print(f"Ensemble Accuracy:  {ens_acc:.3f}")
print(f"Ensemble Latency:   {ens_latency_ms:.3f} ms per prediction")
print(classification_report(y_test, ens_pred, target_names=labels))

ens_pipeline = make_pipeline(
    MinMaxScaler(),
    VotingClassifier(
        estimators=[
            ("knn", KNeighborsClassifier(n_neighbors=5, weights="distance", metric="euclidean")),
            ("rf",  RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_leaf=2, random_state=42)),
            ("svm", SVC(kernel="rbf", C=10, gamma="scale", probability=True, random_state=42)),
            ("mlp", MLPClassifier(hidden_layer_sizes=(128, 64, 32), activation="relu", solver="adam", max_iter=500, early_stopping=True, validation_fraction=0.1, random_state=42))
        ],
        voting="soft"
    )
)
ens_cv = cross_val_score(ens_pipeline, X, y, cv=5, scoring="accuracy")
print(f"Ensemble CV accuracy: {ens_cv.mean():.3f} +/- {ens_cv.std():.3f}")
print(f"Per-fold:             {ens_cv.round(3)}")

cm = confusion_matrix(y_test, ens_pred, labels=labels)
ConfusionMatrixDisplay(cm, display_labels=labels).plot(cmap="Purples")
plt.title("Ensemble Confusion Matrix")
plt.savefig("confusion_matrix_ensemble.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Cell 11: Model Comparison
results = pd.DataFrame({
    "Model":       ["KNN", "Random Forest", "SVM", "MLP", "Ensemble"],
    "Accuracy":    [knn_acc, rf_acc, svm_acc, mlp_acc, ens_acc],
    "CV Mean":     [knn_cv.mean(), rf_cv.mean(), svm_cv.mean(), mlp_cv.mean(), ens_cv.mean()],
    "CV Std":      [knn_cv.std(),  rf_cv.std(),  svm_cv.std(),  mlp_cv.std(),  ens_cv.std()],
    "Latency(ms)": [knn_latency_ms, rf_latency_ms, svm_latency_ms, mlp_latency_ms, ens_latency_ms],
})
results = results.sort_values("CV Mean", ascending=False).reset_index(drop=True)
print(results.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
results.plot(x="Model", y=["Accuracy", "CV Mean"], kind="bar", ax=axes[0])
axes[0].set_title("Accuracy Comparison")
axes[0].set_ylim(0, 1.1)
axes[0].set_xticklabels(results["Model"], rotation=0)
results.plot(x="Model", y="Latency(ms)", kind="bar", ax=axes[1], color="orange")
axes[1].set_title("Inference Latency (ms)")
axes[1].set_xticklabels(results["Model"], rotation=0)
plt.tight_layout()
plt.savefig("model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

best = results.iloc[0]["Model"]
print(f"Best model by CV accuracy: {best}")

In [ ]:
# Cell 12: Save Models
with open("enose_rf_model.pkl", "wb") as f:
    pickle.dump(rf, f)
with open("enose_knn_model.pkl", "wb") as f:
    pickle.dump(knn, f)
with open("enose_svm_model.pkl", "wb") as f:
    pickle.dump(svm, f)
with open("enose_mlp_model.pkl", "wb") as f:
    pickle.dump(mlp, f)
with open("enose_ensemble_model.pkl", "wb") as f:
    pickle.dump(ensemble, f)
with open("enose_scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)
joblib.dump(feature_cols, "feature_columns.pkl")

print("Saved all model files.")

from google.colab import files
files.download("enose_rf_model.pkl")
files.download("enose_knn_model.pkl")
files.download("enose_svm_model.pkl")
files.download("enose_mlp_model.pkl")
files.download("enose_ensemble_model.pkl")
files.download("enose_scaler.pkl")
files.download("feature_columns.pkl")